In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

from cn3s import CN3S, CN3SOptimizer, CN3SParams
from utils import load_data


## Select Basin

In [2]:
BASIN = "RioBranco"
FREQ = "D"  # "D" for daily, "M" for monthly

path, data = load_data(BASIN, freq=FREQ)

In [3]:
data.keys()

dict_keys(['station', 'area', 'prec', 'q', 'shp'])

In [4]:
from cn3s.objectives import Objectives

model = CN3S(CN3SParams(area=data["area"], warmup_steps=1))

if FREQ == "D":
    sliced_prec = data["prec"]["prec"].iloc[-3000:-1000]
    freq = "Daily"
else:
    sliced_prec = data["prec"]["prec"]
    freq = "Monthly"

freq


'Daily'

In [5]:
FREQS = ["Daily", "Monthly"]
FREQS = ["Daily"]

history = {
    freq: {
        step: pd.read_pickle(f"/data/CN3S/basins/{BASIN}/{freq}_history_PlainNSE_{step}_Steps.pkl")
        for step in range(1, 6, 1)
    }
    for freq in FREQS
}


In [8]:
best_history = {
    freq: pd.DataFrame(
        [history[freq][step].iloc[-1] for step in range(1, 6, 1)],
        index=range(1, 6, 1),
    )
    for freq in FREQS
}

In [10]:
best_history["Daily"]

,train_objective,test_objective,train_nse,test_nse,r0,cn_i,alfa,beta,k0,k1,k2,act,params
1,0.246049,None,None,None,50.028455,8.278915,0.689719,0.163228,0.034774,0.349712,0.047600,2,"CN3SParams(name='Unnamed', area=27898.35854781..."
2,0.246087,None,None,None,50.002902,5.498376,0.301902,0.354141,0.373030,0.349910,0.047694,2,"CN3SParams(name='Unnamed', area=27898.35854781..."
3,0.246234,None,None,None,50.027714,2.065622,0.357820,0.250383,0.369582,0.350163,0.047859,2,"CN3SParams(name='Unnamed', area=27898.35854781..."
4,0.246352,None,None,None,50.005933,4.491555,0.791747,0.624953,0.604523,0.349771,0.047819,2,"CN3SParams(name='Unnamed', area=27898.35854781..."
5,0.246348,None,None,None,50.173648,13.991868,0.882807,0.636943,0.349729,0.351000,0.047929,2,"CN3SParams(name='Unnamed', area=27898.35854781..."


In [11]:
df = best_history["Daily"]
df.index.name = "Warmup Steps"
df["train_nse"] = 1 - df["train_objective"]
df

,train_objective,test_objective,train_nse,test_nse,r0,cn_i,alfa,beta,k0,k1,k2,act,params
Warmup Steps,,,,,,,,,,,,,
1,0.246049,None,0.753951,None,50.028455,8.278915,0.689719,0.163228,0.034774,0.349712,0.047600,2,"CN3SParams(name='Unnamed', area=27898.35854781..."
2,0.246087,None,0.753913,None,50.002902,5.498376,0.301902,0.354141,0.373030,0.349910,0.047694,2,"CN3SParams(name='Unnamed', area=27898.35854781..."
3,0.246234,None,0.753766,None,50.027714,2.065622,0.357820,0.250383,0.369582,0.350163,0.047859,2,"CN3SParams(name='Unnamed', area=27898.35854781..."
4,0.246352,None,0.753648,None,50.005933,4.491555,0.791747,0.624953,0.604523,0.349771,0.047819,2,"CN3SParams(name='Unnamed', area=27898.35854781..."
5,0.246348,None,0.753652,None,50.173648,13.991868,0.882807,0.636943,0.349729,0.351000,0.047929,2,"CN3SParams(name='Unnamed', area=27898.35854781..."


In [12]:
BASIN = "RioBranco"
FREQ = "D"  # "D" for daily, "M" for monthly
STEPS = 2

path, data = load_data(BASIN, freq=FREQ)
prec = data["prec"]["prec"].iloc[-3000:-1000]

In [13]:
optim = CN3SOptimizer(
    prec=prec,
    observed_q=data["q"]["Vazao"],
    model=CN3S(CN3SParams(area=data["area"], warmup_steps=STEPS)),
    train_ratio=0.8,
    max_act=10,
    objective=Objectives.PlainNSE,
)

optim.history = history["Daily"][STEPS]

optim.evaluate_step(50)

{'train_objective': 0.24608699798902967,
 'test_objective': 0.5215400484601612,
 'train_nse': 0.7539130020109703,
 'test_nse': 0.4784599515398388}

In [14]:
optim.plot_step(50)

In [16]:
import numpy as np

arr = np.array(optim.model.last_vj)
arr

array([5.        , 4.03694801, 1.77205031, ..., 1.1070957 , 1.03867322,
       1.00321112])

In [20]:
(arr >= 3).sum()

581

In [21]:
arr.shape

(1998,)

# 99 — Scratch: ACT (Average Concentration Time) Verification

This notebook verifies the `act` parameter added to `CN3SParams`, the `pd.Series` path in `CN3S.run()`, and the `CN3SOptimizer` integration.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

from cn3s import CN3S, CN3SOptimizer, CN3SParams

## 1 — CN3SParams: round-trip test

In [ ]:
# as_list() should now return 8 elements
p = CN3SParams(area=9642.0, act=4)
vec = p.as_list()
print(f"as_list length: {len(vec)}  (expected 8)")
print(f"vec[-1] = {vec[-1]}  (expected 4.0)")
assert len(vec) == 8
assert vec[-1] == 4.0

# from_vector should round act to int
p2 = CN3SParams.from_vector(vec, area=9642.0)
print(f"p2.act = {p2.act!r}  (expected 4, type int)")
assert p2.act == 4
assert isinstance(p2.act, int)

# Float rounding: 4.6 → 5
vec_round = list(vec)
vec_round[-1] = 4.6
p3 = CN3SParams.from_vector(vec_round, area=9642.0)
print(f"p3.act (from 4.6) = {p3.act!r}  (expected 5)")
assert p3.act == 5

print("\nAll CN3SParams checks PASSED")

## 2 — CN3S.run() with a pd.Series: index is set automatically

In [ ]:
import numpy as np

ACT = 4
N = 100

# Build a synthetic dated precipitation Series
dates = pd.date_range("2010-01-01", periods=N, freq="D")
prec_series = pd.Series(np.random.default_rng(42).uniform(0, 10, size=N), index=dates, name="prec")

params = CN3SParams(area=9642.0, act=ACT)
model = CN3S(params)

model.run(prec_series)  # pd.Series path

expected_first_date = dates[3] + pd.Timedelta(days=ACT)
actual_first_date = model.results.index[0]

print(f"Expected results.index[0]: {expected_first_date.date()}")
print(f"Actual  results.index[0]: {actual_first_date.date()}")
assert actual_first_date == expected_first_date, "Index mismatch!"
print(f"results shape: {model.results.shape}  (expected ({N - 3}, 11))")
assert model.results.shape == (N - 3, 11)

model.results.head()

## 3 — CN3S.run() with a plain list: RangeIndex preserved

In [ ]:
model_list = CN3S(CN3SParams(area=9642.0, act=ACT))
model_list.run(prec_series.tolist())  # list path — no index management

print(f"results.index type: {type(model_list.results.index).__name__}  (expected RangeIndex)")
assert isinstance(model_list.results.index, pd.RangeIndex), "Expected RangeIndex for list input"
print("List-path check PASSED")

## 4 — Load real data (requires /data/CN3S mount)

In [ ]:
BASIN = "BarraDosBugres"
STATION = 66010000
BASE_FOLDER = "/data/CN3S/basins/"

path = Path(BASE_FOLDER) / BASIN
assert path.exists(), f"Path {path} does not exist — skipping real-data cells"

prec = pd.read_parquet(path / f"Daily_Prec_{STATION}.parquet")
prec.index = prec.index - pd.Timedelta(hours=12)  # mandatory 12h correction

q = pd.read_parquet(path / f"Daily_Q_{STATION}.parquet")

print(f"prec: {len(prec)} rows, {prec.index[0].date()} → {prec.index[-1].date()}")
print(f"q   : {len(q)} rows, {q.index[0].date()} → {q.index[-1].date()}")

## 5 — CN3S.run() with real dated series (act=4)

In [ ]:
AREA = 9642  # km²
ACT = 4

model_real = CN3S(CN3SParams(area=AREA, act=ACT))
model_real.run(prec["prec"])  # pd.Series — index auto-assigned with ACT shift

print(f"results.index[0] : {model_real.results.index[0].date()}")
print(f"prec.index[3]+4d : {(prec.index[3] + pd.Timedelta(days=ACT)).date()}")
assert model_real.results.index[0] == prec.index[3] + pd.Timedelta(days=ACT)

model_real.results.head()

## 6 — CN3SOptimizer: verify act is optimized (short run)

In [ ]:
sliced_prec = prec.iloc[-1000:-500]

model_opt = CN3S(CN3SParams(area=AREA))
optim = CN3SOptimizer(
    prec=sliced_prec["prec"],
    observed_q=q["Vazao"],
    model=model_opt,
    train_ratio=0.7,
    max_act=10,
)

print(f"Bounds count: {len(optim._bounds_list)}  (expected 8)")
print(f"act bound: {optim._bounds_list[-1]}  (expected (0.0, 10.0))")
assert len(optim._bounds_list) == 8
assert optim._bounds_list[-1] == (0.0, 10.0)

print(f"Train cutoff date: {optim._train_cutoff.date()}")
print(f"Aligned record: {len(optim.aligned)} rows")

In [ ]:
# Short optimization run — just 5 iterations to verify end-to-end flow
optim.optimize(maxiter=5, tol=1e-2, disp=False, workers=1)

In [ ]:
print(f"Optimized params: {optim.model.params}")
print(f"act type: {type(optim.model.params.act).__name__}  (expected int)")
assert isinstance(optim.model.params.act, int)
print(f"results index type: {type(optim.model.results.index).__name__}")
print(f"results.index[0]: {optim.model.results.index[0].date()}")
print("\nOptimizer ACT integration PASSED")

In [ ]:
optim.model.results.head()